In [19]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.figure_factory as ff
from collections import Counter
from rdkit import Chem
from rdkit.Chem import Descriptors
import networkx as nx


In [20]:
from pathlib import Path
import pandas as pd

# 1) Define the base folder *relative* to your notebook/script
current_Path = Path.cwd()
BASE = current_Path.parent.parent

# 1) Point to the train & val sub-folders
train_dir = BASE / "data_curation" / "original_curated_with_embeddings_and_MW" / "train_without_data_augmentation"
val_dir   = BASE / "data_curation" / "original_curated_with_embeddings_and_MW" / "val"
test_dir = BASE / "data_curation" / "original_curated_with_embeddings_and_MW" / "test_predictions"

# 2) Glob and sort
train_files = sorted(train_dir.glob("train*_curated.csv"))
val_files   = sorted(val_dir.glob("val*_curated.csv"))
test_file =  BASE / "data_curation" / "original_curated_with_embeddings_and_MW" / "test_predictions" / "consensus_without_data_augmentation.csv"

# 3) Load all training files into a list 
train_dfs = []
for file in train_files: 
    train_df = pd.read_csv(file)
    train_dfs.append(train_df)

# 4) Load all val  files into a list 
val_dfs = []
for file in val_files: 
    val_df = pd.read_csv(file)
    val_dfs.append(val_df)

#5) Combine all the datasets into a final train df and drop SMILES duplicates 
train_combined_df = pd.concat(train_dfs, ignore_index=True)
val_combined_df = pd.concat(val_dfs, ignore_index=True)
final_train_df = pd.concat([train_combined_df, val_combined_df], ignore_index=True)
print ("Before dropping duplicates: " + str(len(final_train_df)))
final_train_df = final_train_df.drop_duplicates(subset="SMILES")
print ("After dropping duplicates: " + str(len(final_train_df)))

#6) View df
final_train_df.info()
print(final_train_df.isna().sum())
print("Number of unique SMILES values: " + str(final_train_df["SMILES"].nunique()))

# 7) Save the final train df
final_train_df.to_csv(BASE / "data_curation" / "original_curated_with_embeddings_and_MW" / "train_without_data_augmentation" / "final_train_df.csv", index=False)


Before dropping duplicates: 176590
After dropping duplicates: 17633
<class 'pandas.core.frame.DataFrame'>
Index: 17633 entries, 0 to 89122
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   SMILES      17633 non-null  object 
 1   MP          17633 non-null  float64
 2   embeddings  17633 non-null  object 
 3   MW          17633 non-null  float64
dtypes: float64(2), object(2)
memory usage: 688.8+ KB
SMILES        0
MP            0
embeddings    0
MW            0
dtype: int64
Number of unique SMILES values: 17633


In [21]:
from pathlib import Path
import pandas as pd
import re

# --- Helpers ---
def standardize_smiles_column(df: pd.DataFrame) -> pd.DataFrame:
    if "SMILES" not in df.columns:
        candidates = [c for c in df.columns if "smile" in c.lower()]
        if not candidates:
            raise ValueError(f"No SMILES-like column found. Columns: {list(df.columns)}")
        df = df.rename(columns={candidates[0]: "SMILES"})
    df["SMILES"] = df["SMILES"].astype(str).str.strip().replace({"": pd.NA})
    return df

def standardize_mp_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename exp MP / Experimental_MP → MP and coerce to numeric.
    Uses a strict regex so we don't accidentally match 'sample', etc.
    """
    mp_cols = [c for c in df.columns
               if re.fullmatch(r"\s*(exp[\s_]+)?mp\s*", c, flags=re.IGNORECASE)]
    if len(mp_cols) == 0:
        # no MP in this frame is OK for presence-only tables
        return df
    # prefer 'exp MP' if both exist (unlikely), else the first match
    mp_col = sorted(mp_cols, key=lambda c: ("exp" not in c.lower(), c.lower()))[0]
    if mp_col != "MP":
        df = df.rename(columns={mp_col: "MP"})
    # make numeric (non-numeric → NaN)
    df["MP"] = pd.to_numeric(df["MP"], errors="coerce")
    return df

def standardize_cols(df: pd.DataFrame) -> pd.DataFrame:
    return standardize_mp_column(standardize_smiles_column(df.copy()))

# --- Read & standardize ---
test_df  = pd.read_csv(test_file)
train_all = standardize_cols(train_combined_df)   # from your earlier code
val_all   = standardize_cols(val_combined_df)
test_all  = standardize_cols(test_df)             # converts 'exp MP' → 'MP'

# --- Build tidy long table with targets ---
train_long = train_all[["SMILES", "MP"]].assign(split="train") if "MP" in train_all else train_all[["SMILES"]].assign(MP=pd.NA, split="train")
val_long   = val_all[["SMILES", "MP"]].assign(split="val")     if "MP" in val_all   else val_all[["SMILES"]].assign(MP=pd.NA,   split="val")
test_long  = test_all[["SMILES", "MP"]].assign(split="test")   if "MP" in test_all  else test_all[["SMILES"]].assign(MP=pd.NA,  split="test")

all_long = pd.concat([train_long, val_long, test_long], ignore_index=True)
all_long = all_long.dropna(subset=["SMILES"])

# --- Presence matrix (no targets) ---
presence = (
    all_long.assign(present=True)
    .pivot_table(index="SMILES", columns="split", values="present", aggfunc="any", fill_value=False)
    .reset_index()
)

# --- Targets per split (MP pivot) ---
# If duplicates exist for (SMILES, split), take median (robust). Change to 'mean' or 'first' as you prefer.
targets = (
    all_long.dropna(subset=["MP"])
            .groupby(["SMILES", "split"], as_index=False)["MP"].median()
            .pivot(index="SMILES", columns="split", values="MP")
            .reset_index()
)

# --- Quick stats ---
n_train = presence["train"].sum() if "train" in presence else 0
n_val   = presence["val"].sum()   if "val" in presence   else 0
n_test  = presence["test"].sum()  if "test" in presence  else 0

overlap_tv = int((presence.get("train", False) & presence.get("val", False)).sum())
overlap_tt = int((presence.get("train", False) & presence.get("test", False)).sum())
overlap_vt = int((presence.get("val",   False) & presence.get("test", False)).sum())

print(f"Unique SMILES — train: {n_train}, val: {n_val}, test: {n_test}")
print(f"Overlaps — train∩val: {overlap_tv}, train∩test: {overlap_tt}, val∩test: {overlap_vt}")

# --- Unique list of SMILES ---
all_unique_smiles = presence[["SMILES"]].copy()
print(f"Total unique SMILES across all splits: {len(all_unique_smiles)}")

# --- Save outputs ---
cache_dir = BASE / "data_curation" / "original_curated_with_embeddings_and_MW" / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)

presence.to_csv(cache_dir / "smiles_presence_train_val_test.csv", index=False)
all_unique_smiles.to_csv(cache_dir / "all_unique_smiles.csv", index=False)
targets.to_csv(cache_dir / "mp_targets_per_split.csv", index=False)

print("Saved:\n - smiles_presence_train_val_test.csv\n - all_unique_smiles.csv\n - mp_targets_per_split.csv")

df = all_long

Unique SMILES — train: 17633, val: 16193, test: 1961
Overlaps — train∩val: 16193, train∩test: 6, val∩test: 5
Total unique SMILES across all splits: 19588
Saved:
 - smiles_presence_train_val_test.csv
 - all_unique_smiles.csv
 - mp_targets_per_split.csv


In [22]:
# Extract unique, non-null SMILES
unique_smiles = pd.unique(df['SMILES'].values.ravel())
unique_smiles = [s for s in unique_smiles if pd.notnull(s)]

# Build dynamic descriptor list
descriptor_funcs = [(name, func) for name, func in Descriptors.descList]
descriptor_names = [name for name, _ in descriptor_funcs]

print(f"✅ {len(unique_smiles)} unique SMILES found.")
print(f"✅ {len(descriptor_names)} descriptors will be computed for each SMILES.")

✅ 19588 unique SMILES found.
✅ 217 descriptors will be computed for each SMILES.


In [23]:
# Function to compute descriptors for a single SMILES
def calc_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [np.nan] * len(descriptor_funcs)
    return [func(mol) for _, func in descriptor_funcs]

# Compute descriptors for all unique SMILES
desc_dict = {}
for smi in unique_smiles:
    desc_dict[smi] = calc_descriptors(smi)

# Convert to DataFrame
desc_df = pd.DataFrame.from_dict(desc_dict, orient="index", columns=descriptor_names)
desc_df.index.name = "SMILES"

print("Descriptor DataFrame created with shape:", desc_df.shape)
desc_df.head()


Descriptor DataFrame created with shape: (19588, 217)


,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
SMILES,,,,,,,,,,,,,,,,,,,,,
FC1(F)C(F)(F)C(F)(F)C(C2(C1(F)C1(F)C(F)(F)C(F)(F)C(C(C1(F)C(C2(F)F)(F)F)(F)F)(F)F)F)(F)F,15.314375,15.314375,8.764932,-9.769375,0.250251,47.368421,624.106,624.106,623.961677,224,...,0,0,0,0,0,0,0,0,0,0
CCc1ncc[nH]1,3.979167,3.979167,0.993056,0.993056,0.554141,9.285714,96.133,88.069,96.068748,38,...,0,0,0,0,0,0,0,0,0,0
S=C(N(C)C)Sc1ccc2c(c1)cccc2,5.271563,5.271563,0.884520,0.884520,0.557540,10.375000,247.388,234.284,247.048941,82,...,1,0,0,0,0,0,0,0,0,0
Nc1nc(N)nc(n1)c1ccccc1,5.454549,5.454549,0.135512,0.135512,0.685396,10.000000,187.206,178.134,187.085795,70,...,0,0,0,0,0,0,0,0,0,0
NCCNc1ccc(cn1)[N+](=O)[O-],10.251416,10.251416,0.015464,-0.486397,0.517531,9.615385,182.183,172.103,182.080376,70,...,0,0,0,0,0,0,0,0,0,0


In [29]:
import numpy as np
import pandas as pd

# 1) Clean infinities (rare) → NaN
desc_df = desc_df.replace([np.inf, -np.inf], np.nan)

# 2) Drop columns that contain any NaN values
nan_counts = desc_df.isna().sum()
any_nan_col = nan_counts[nan_counts > 0].index.tolist()
print(f'Dropping {len(any_nan_col)} descriptors with partial or full NaN values.')

desc_df = desc_df.drop(columns=any_nan_col, errors='ignore')

# 3) Compute variance on the remaining columns
col_var = desc_df.var(axis=0, ddof=0, numeric_only=True)


# 4) Identify columns to drop: exactly zero variance
zero_var_cols = col_var.index[(col_var <= 0.0) | (col_var.isna())].tolist()
print(f"Dropping {len(zero_var_cols)}")

# 4) Drop them
desc_df_reduced = desc_df.drop(columns=zero_var_cols, errors="ignore")

print(f"Dropped {len(zero_var_cols)} zero-variance.")
print("Kept columns:", desc_df_reduced.shape[1])

# Save artifacts
pd.Series(zero_var_cols, name="dropped_zero_variance").to_csv("dropped_zero_variance_features.csv", index=False)
desc_df_reduced.to_csv("rdkit_descriptors_zeroVarDropped.csv")

kept_cols = desc_df_reduced.columns.tolist()
pd.Series(kept_cols, name='kept_descriptors').to_csv('kept_descriptors.csv', index=False)

Dropping 0 descriptors with partial or full NaN values.
Dropping 3
Dropped 3 zero-variance.
Kept columns: 202


In [31]:
import pandas as pd

# load your reduced descriptors
rdk_df = pd.read_csv("rdkit_descriptors_zeroVarDropped.csv")

# ignore meta columns
meta_cols = {"SMILES", "MP", "split"}
rdkit_cols = [c for c in rdk_df.columns if c not in meta_cols]

# stats per column
nan_counts = rdk_df[rdkit_cols].isna().sum()
total_rows = len(rdk_df)

# classify
all_nan_cols = nan_counts[nan_counts == total_rows].index.tolist()
partial_nan_cols = nan_counts[(nan_counts > 0) & (nan_counts < total_rows)].index.tolist()

print(f"Total descriptors: {len(rdkit_cols)}")
print(f"Columns with all NaN: {len(all_nan_cols)}")
print(f"Columns with partial NaN: {len(partial_nan_cols)}")

# optional: show top offenders
if partial_nan_cols:
    top10 = nan_counts[partial_nan_cols].sort_values(ascending=False).head(10)
    print("\nTop 10 partial-NaN descriptors:")
    print(top10)


Total descriptors: 202
Columns with all NaN: 0
Columns with partial NaN: 0


In [32]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

def _load_rdkit_desc(rdkit_csv_path: str, smiles_col: str = "SMILES") -> pd.DataFrame:
    """
    Load the RDKit descriptor table saved earlier. Be robust to whether SMILES
    was saved as an index or a regular column.
    """
    desc = pd.read_csv(rdkit_csv_path)

    # If SMILES isn't a column, try to recover it from the index or first column.
    if smiles_col not in desc.columns:
        # If index carries SMILES, reset it
        if desc.index.name == smiles_col:
            desc = desc.reset_index()
        else:
            # Otherwise assume first column is SMILES (common when index was saved unnamed)
            first_col = desc.columns[0]
            if first_col != smiles_col:
                desc = desc.rename(columns={first_col: smiles_col})

    # Ensure one row per SMILES
    if desc.duplicated(subset=[smiles_col]).any():
        desc = desc.drop_duplicates(subset=[smiles_col], keep="first")

    return desc

def build_fold_data_rdkit_only(
    df: pd.DataFrame,
    rdkit_csv_path: str = "rdkit_descriptors_zeroVarDropped.csv",
    smiles_col: str = "SMILES",
    target_col: str = "MP",
    fold_col: str = "cv_fold",
    scaler_cls = StandardScaler,
):
    """
    Prepare per-fold train/val matrices using ONLY RDKit descriptors.
    - Merges your main df (with SMILES/MP/cv_fold) to a precomputed descriptor table.
    - Replaces ±inf→NaN, imputes NaNs with *training medians* (per fold).
    - Scales descriptors with a single StandardScaler fit on the training split.
    Returns: dict[fold] -> {X_train, y_train, X_val, y_val, scaler, feature_names}
    """

    # 0) Load descriptors and merge onto main df by SMILES
    desc = _load_rdkit_desc(rdkit_csv_path, smiles_col=smiles_col)
    rdkit_cols = [c for c in desc.columns if c != smiles_col]  # all descriptor columns
    dfm = df.merge(desc, on=smiles_col, how="left")

    # 1) Number of folds (assumes 0..K-1 labels)
    K = int(dfm[fold_col].max()) + 1
    fold_data = {}

    for k in range(K):
        tr = dfm[dfm[fold_col] != k].copy()
        va = dfm[dfm[fold_col] == k].copy()

        # 2) Clean RDKit: replace infs -> NaN
        tr_rd = tr[rdkit_cols].replace([np.inf, -np.inf], np.nan)
        va_rd = va[rdkit_cols].replace([np.inf, -np.inf], np.nan)

        # 3) Compute training medians (numeric) and impute NaNs in train/val with those medians
        med = tr_rd.median(numeric_only=True)

        # If any column is all-NaN in training, med will be NaN — drop such columns for this fold
        bad_cols = med.index[med.isna()].tolist()
        if bad_cols:
            print(f"[Fold {k}] Dropping {len(bad_cols)} all-NaN-in-train columns for this fold.")
            keep_cols = [c for c in rdkit_cols if c not in bad_cols]
            tr_rd = tr_rd[keep_cols]
            va_rd = va_rd[keep_cols]
            med = med.drop(bad_cols)
        else:
            keep_cols = rdkit_cols

        tr_rd = tr_rd.fillna(med)
        va_rd = va_rd.fillna(med)

        # 4) To numpy (float32)
        X_tr = tr_rd.to_numpy(dtype=np.float32)
        X_va = va_rd.to_numpy(dtype=np.float32)

        # 5) Scale descriptors (fit on train only)
        scaler = scaler_cls().fit(X_tr)
        X_train = scaler.transform(X_tr).astype(np.float32)
        X_val   = scaler.transform(X_va).astype(np.float32)

        # 6) Targets
        y_train = tr[target_col].to_numpy(np.float32)
        y_val   = va[target_col].to_numpy(np.float32)

        # 7) Sanity checks
        for name, arr in [("X_train", X_train), ("X_val", X_val), ("y_train", y_train), ("y_val", y_val)]:
            if not np.isfinite(arr).all():
                raise ValueError(f"[Fold {k}] {name} has NaN/Inf")

        # 8) Package
        fold_data[k] = dict(
            X_train=X_train, y_train=y_train,
            X_val=X_val,     y_val=y_val,
            scaler=scaler,
            feature_names=keep_cols,
        )

        print(f"[Fold {k}] X_train {X_train.shape} | X_val {X_val.shape} | RDKit_dim={len(keep_cols)}")

    return fold_data
